# Dealer Debt Price Analysis

This notebook visualizes how debt prices evolve over time and across maturity buckets
in dealer-intermediated Kalecki ring simulations.

**Sections:**
1. Setup — imports, parameters, run simulation
2. Data Extraction — convert metrics into plottable structures
3. VWAP by Bucket — volume-weighted average trade price vs VBT mid and dealer midline
4. Term Structure & Price-Default — cross-bucket price curve evolution + defaults overlay
5. Supplementary — trade volume, bid-ask spreads, inventory-price relationship, summary stats

In [ ]:
# Cell 1: Imports and Parameters
import warnings
warnings.filterwarnings('ignore')

from decimal import Decimal
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy import stats

from bilancio.config.models import (
    ScenarioConfig,
    RingExplorerGeneratorConfig,
)
from bilancio.config.loaders import preprocess_config
from bilancio.config.apply import apply_to_system
from bilancio.scenarios.ring_explorer import compile_ring_explorer_balanced
from bilancio.engines.system import System
from bilancio.engines.simulation import run_until_stable

# Plotting constants
BUCKET_COLORS = {'short': '#e74c3c', 'mid': '#3498db', 'long': '#2ecc71'}
BUCKETS = ['short', 'mid', 'long']

plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

In [ ]:
# Cell 2: Build scenario programmatically
N_AGENTS = 30
KAPPA = Decimal('0.4')        # stressed liquidity
MATURITY_DAYS = 10
OUTSIDE_MID_RATIO = Decimal('0.85')
FACE_VALUE = Decimal('20')
SEED = 42
MAX_DAYS = 35
QUIET_DAYS = 11
Q_TOTAL = N_AGENTS * FACE_VALUE  # total dues on day 1

# Build generator config
generator_data = {
    'version': 1,
    'generator': 'ring_explorer_v1',
    'name_prefix': 'Price Analysis',
    'params': {
        'n_agents': N_AGENTS,
        'seed': SEED,
        'kappa': str(KAPPA),
        'Q_total': str(Q_TOTAL),
        'inequality': {
            'scheme': 'dirichlet',
            'concentration': '1',
            'monotonicity': '0',
        },
        'maturity': {
            'days': MATURITY_DAYS,
            'mode': 'lead_lag',
            'mu': '0.5',
        },
    },
    'compile': {'emit_yaml': False},
}

generator_config = RingExplorerGeneratorConfig.model_validate(generator_data)

# Compile balanced scenario with active dealer
scenario = compile_ring_explorer_balanced(
    generator_config,
    face_value=FACE_VALUE,
    outside_mid_ratio=OUTSIDE_MID_RATIO,
    mode='active',
    rollover_enabled=True,
)

# Add dealer config
scenario['dealer'] = {
    'enabled': True,
    'ticket_size': 1,
    'dealer_share': str(Decimal('0.25')),
    'vbt_share': str(Decimal('0.50')),
    'risk_assessment': {'enabled': True},
}
scenario['balanced_dealer'] = {
    'enabled': True,
    'face_value': str(FACE_VALUE),
    'outside_mid_ratio': str(OUTSIDE_MID_RATIO),
    'vbt_share_per_bucket': '0.25',
    'dealer_share_per_bucket': '0.125',
    'mode': 'active',
    'rollover_enabled': True,
    'kappa': str(KAPPA),
}

# Set run config
scenario.setdefault('run', {}).update({
    'max_days': MAX_DAYS,
    'quiet_days': QUIET_DAYS,
    'default_handling': 'expel-agent',
})

print(f'Scenario compiled: {N_AGENTS} agents, kappa={KAPPA}, maturity={MATURITY_DAYS}d')
print(f'  Q_total: {Q_TOTAL}, face_value: {FACE_VALUE}')
print(f'  Agents: {len(scenario.get("agents", []))}')
print(f'  Initial actions: {len(scenario.get("initial_actions", []))}')

In [ ]:
# Cell 3: Run the simulation
preprocessed = preprocess_config(scenario)
config = ScenarioConfig.model_validate(preprocessed)

system = System(default_mode='expel-agent')
apply_to_system(config, system)
system.state.rollover_enabled = True

# Stage scheduled actions
if getattr(config, 'scheduled_actions', None):
    for sa in config.scheduled_actions:
        system.state.scheduled_actions_by_day.setdefault(sa.day, []).append(sa.action)

print(f'Running simulation (max {MAX_DAYS} days, quiet {QUIET_DAYS})...')
reports = run_until_stable(system, max_days=MAX_DAYS, quiet_days=QUIET_DAYS, enable_dealer=True)

# Extract metrics
metrics = system.state.dealer_subsystem.metrics
n_trades = len(metrics.trades)
n_snapshots = len(metrics.dealer_snapshots)
n_events = len(system.state.events)
n_defaults = sum(1 for e in system.state.events if e.get('kind') == 'AgentDefaulted')
last_day = system.state.day

print(f'Simulation complete: {last_day} days')
print(f'  Trades: {n_trades} ({sum(1 for t in metrics.trades if t.side=="BUY")} buys, '
      f'{sum(1 for t in metrics.trades if t.side=="SELL")} sells)')
print(f'  Dealer snapshots: {n_snapshots}')
print(f'  Events: {n_events} ({n_defaults} agent defaults)')

In [ ]:
# Cell 4: Data Extraction — convert raw metrics into plottable structures

# Trades by day and bucket
trades_by_day_bucket = defaultdict(lambda: defaultdict(list))
for t in metrics.trades:
    trades_by_day_bucket[t.day][t.bucket].append(t)

# Snapshots by day and bucket
snapshots_by_day_bucket = defaultdict(dict)
for s in metrics.dealer_snapshots:
    snapshots_by_day_bucket[s.day][s.bucket] = s

# Time series from snapshots
all_days = sorted(set(s.day for s in metrics.dealer_snapshots))

vbt_mid_ts = defaultdict(dict)       # vbt_mid_ts[day][bucket] = float
dealer_mid_ts = defaultdict(dict)     # dealer_mid_ts[day][bucket] = float
dealer_bid_ts = defaultdict(dict)
dealer_ask_ts = defaultdict(dict)
spread_ts = defaultdict(dict)
inventory_ts = defaultdict(dict)

for s in metrics.dealer_snapshots:
    vbt_mid_ts[s.day][s.bucket] = float(s.vbt_mid) if s.vbt_mid is not None else None
    dealer_mid_ts[s.day][s.bucket] = float(s.midline) if s.midline is not None else None
    dealer_bid_ts[s.day][s.bucket] = float(s.bid) if s.bid is not None else None
    dealer_ask_ts[s.day][s.bucket] = float(s.ask) if s.ask is not None else None
    spread_ts[s.day][s.bucket] = float(s.spread) if s.spread is not None else None
    inventory_ts[s.day][s.bucket] = float(s.inventory) if s.inventory is not None else None

# Default events by day
defaults_by_day = defaultdict(int)
for e in system.state.events:
    if e.get('kind') == 'AgentDefaulted':
        defaults_by_day[e.get('day', 0)] += 1

cumulative_defaults = {}
running = 0
for d in range(0, last_day + 1):
    running += defaults_by_day.get(d, 0)
    cumulative_defaults[d] = running

# VWAP computation helper
def compute_vwap(trade_list):
    """Volume-weighted average price from a list of TradeRecords."""
    total_value = sum(float(t.unit_price) * float(t.face_value) for t in trade_list)
    total_volume = sum(float(t.face_value) for t in trade_list)
    return total_value / total_volume if total_volume > 0 else None

print(f'Data extracted:')
print(f'  Snapshot days: {len(all_days)} ({min(all_days) if all_days else "N/A"} to {max(all_days) if all_days else "N/A"})')
print(f'  Trade days: {len(trades_by_day_bucket)}')
print(f'  Default days: {len(defaults_by_day)} (total agents: {running})')

## A. VWAP by Bucket Over Time

Volume-weighted average trade price for each maturity bucket, compared against
VBT mid (fair-value benchmark) and dealer midline (market-making quote midpoint).

In [ ]:
# Cell 6: VWAP by Bucket — 3 panel figure
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle('VWAP by Maturity Bucket', fontsize=14, fontweight='bold')

trade_days = sorted(trades_by_day_bucket.keys())

for idx, bucket in enumerate(BUCKETS):
    ax = axes[idx]
    color = BUCKET_COLORS[bucket]

    # VBT mid line
    vbt_days = [d for d in all_days if vbt_mid_ts.get(d, {}).get(bucket) is not None]
    vbt_vals = [vbt_mid_ts[d][bucket] for d in vbt_days]
    if vbt_days:
        ax.plot(vbt_days, vbt_vals, 'k--', linewidth=1.5, label='VBT mid', alpha=0.7)

    # Dealer midline
    dm_days = [d for d in all_days if dealer_mid_ts.get(d, {}).get(bucket) is not None]
    dm_vals = [dealer_mid_ts[d][bucket] for d in dm_days]
    if dm_days:
        ax.plot(dm_days, dm_vals, color=color, linewidth=1.5, label='Dealer mid', alpha=0.8)

    # VWAP scatter - all trades, buys, sells
    vwap_all_days, vwap_all_vals = [], []
    vwap_buy_days, vwap_buy_vals = [], []
    vwap_sell_days, vwap_sell_vals = [], []

    for d in trade_days:
        bucket_trades = trades_by_day_bucket[d].get(bucket, [])
        if not bucket_trades:
            continue
        vwap = compute_vwap(bucket_trades)
        if vwap is not None:
            vwap_all_days.append(d)
            vwap_all_vals.append(vwap)

        buys = [t for t in bucket_trades if t.side == 'BUY']
        sells = [t for t in bucket_trades if t.side == 'SELL']
        buy_vwap = compute_vwap(buys) if buys else None
        sell_vwap = compute_vwap(sells) if sells else None

        if buy_vwap is not None:
            vwap_buy_days.append(d)
            vwap_buy_vals.append(buy_vwap)
        if sell_vwap is not None:
            vwap_sell_days.append(d)
            vwap_sell_vals.append(sell_vwap)

    if vwap_all_days:
        ax.scatter(vwap_all_days, vwap_all_vals, c='gray', s=30, zorder=5,
                   alpha=0.5, label='VWAP (all)')
    if vwap_buy_days:
        ax.scatter(vwap_buy_days, vwap_buy_vals, marker='^', c='green', s=40,
                   zorder=6, alpha=0.7, label='VWAP (buys)')
    if vwap_sell_days:
        ax.scatter(vwap_sell_days, vwap_sell_vals, marker='v', c='red', s=40,
                   zorder=6, alpha=0.7, label='VWAP (sells)')

    if not vwap_all_days and not vbt_days and not dm_days:
        ax.text(0.5, 0.5, 'No trades recorded', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='gray')

    ax.set_title(f'{bucket.capitalize()} bucket', fontweight='bold')
    ax.set_xlabel('Day')
    if idx == 0:
        ax.set_ylabel('Price (per unit face)')
    ax.legend(fontsize=8, loc='lower left')

plt.tight_layout()
plt.show()

## B. Term Structure & Price-Default Dynamics

Left: How the dealer midline term structure (short → mid → long) evolves over time.  
Right: Heatmap of dealer midline prices across buckets and days.

In [ ]:
# Cell 8: Term Structure Evolution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Term Structure Evolution', fontsize=14, fontweight='bold')

# Left panel: overlaid term structure curves at selected days
if all_days:
    n_curves = min(8, len(all_days))
    selected_days = [all_days[int(i * (len(all_days) - 1) / max(1, n_curves - 1))]
                     for i in range(n_curves)]
    # Deduplicate while preserving order
    seen = set()
    selected_days = [d for d in selected_days if not (d in seen or seen.add(d))]

    viridis = plt.get_cmap('viridis', len(selected_days))

    for i, d in enumerate(selected_days):
        vals = [dealer_mid_ts.get(d, {}).get(b) for b in BUCKETS]
        if all(v is not None for v in vals):
            ax1.plot(BUCKETS, vals, 'o-', color=viridis(i), linewidth=2,
                     markersize=6, label=f'Day {d}')

    ax1.set_xlabel('Maturity Bucket')
    ax1.set_ylabel('Dealer Midline')
    ax1.set_title('Term Structure at Selected Days')
    ax1.legend(fontsize=7, loc='lower left', ncol=2)
else:
    ax1.text(0.5, 0.5, 'No snapshot data', transform=ax1.transAxes,
             ha='center', va='center', fontsize=12, color='gray')

# Right panel: heatmap of dealer midline
if all_days:
    heatmap_data = []
    for bucket in BUCKETS:
        row = []
        for d in all_days:
            val = dealer_mid_ts.get(d, {}).get(bucket)
            row.append(val if val is not None else np.nan)
        heatmap_data.append(row)

    heatmap_arr = np.array(heatmap_data, dtype=float)

    # Only plot if we have non-NaN values
    if not np.all(np.isnan(heatmap_arr)):
        im = ax2.imshow(heatmap_arr, aspect='auto', cmap='RdYlGn',
                        interpolation='nearest')
        ax2.set_yticks(range(len(BUCKETS)))
        ax2.set_yticklabels([b.capitalize() for b in BUCKETS])

        # Show subset of day labels
        n_labels = min(10, len(all_days))
        tick_positions = [int(i * (len(all_days) - 1) / max(1, n_labels - 1))
                          for i in range(n_labels)]
        ax2.set_xticks(tick_positions)
        ax2.set_xticklabels([str(all_days[p]) for p in tick_positions])

        ax2.set_xlabel('Day')
        ax2.set_title('Dealer Midline Heatmap')
        plt.colorbar(im, ax=ax2, label='Price')
    else:
        ax2.text(0.5, 0.5, 'All NaN values', transform=ax2.transAxes,
                 ha='center', va='center', fontsize=12, color='gray')
else:
    ax2.text(0.5, 0.5, 'No snapshot data', transform=ax2.transAxes,
             ha='center', va='center', fontsize=12, color='gray')

plt.tight_layout()
plt.show()

### C. Price-Default Relationship

VBT mid price curves overlaid with cumulative defaults. Vertical red lines mark days
when agents defaulted.

In [ ]:
# Cell 10: Price-Default Relationship
fig, ax1 = plt.subplots(figsize=(12, 5))
fig.suptitle('Price-Default Relationship', fontsize=14, fontweight='bold')

# VBT mid curves per bucket on left axis
has_vbt_data = False
for bucket in BUCKETS:
    days_b = [d for d in all_days if vbt_mid_ts.get(d, {}).get(bucket) is not None]
    vals_b = [vbt_mid_ts[d][bucket] for d in days_b]
    if days_b:
        ax1.plot(days_b, vals_b, color=BUCKET_COLORS[bucket], linewidth=2,
                 label=f'VBT mid ({bucket})')
        has_vbt_data = True

ax1.set_xlabel('Day')
ax1.set_ylabel('VBT Mid Price', color='black')
ax1.tick_params(axis='y', labelcolor='black')

# Cumulative defaults on right axis
ax2 = ax1.twinx()
cum_days = sorted(cumulative_defaults.keys())
cum_vals = [cumulative_defaults[d] for d in cum_days]
max_defaults = max(cum_vals) if cum_vals else 0

if max_defaults > 0:
    ax2.fill_between(cum_days, cum_vals, alpha=0.15, color='red', label='Cumulative defaults')
    ax2.plot(cum_days, cum_vals, color='red', linewidth=1, alpha=0.5)

    # Vertical lines on default days
    for d, count in defaults_by_day.items():
        ax2.axvline(x=d, color='red', alpha=0.3, linewidth=0.8, linestyle=':')

ax2.set_ylabel('Cumulative Defaults', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left', fontsize=8)

plt.tight_layout()
plt.show()

# Pearson correlation between average VBT mid and cumulative defaults
if max_defaults > 0 and has_vbt_data:
    # Average VBT mid across buckets for correlation
    common_days = [d for d in cum_days if d in vbt_mid_ts]
    if len(common_days) >= 3:
        avg_vbt = []
        cum_def = []
        for d in common_days:
            bucket_vals = [vbt_mid_ts[d].get(b) for b in BUCKETS
                           if vbt_mid_ts[d].get(b) is not None]
            if bucket_vals:
                avg_vbt.append(np.mean(bucket_vals))
                cum_def.append(cumulative_defaults[d])

        if len(set(cum_def)) > 1 and len(avg_vbt) >= 3:
            r, p = stats.pearsonr(avg_vbt, cum_def)
            print(f'Pearson r(avg VBT mid, cumulative defaults) = {r:.3f} (p={p:.4f})')
        else:
            print('Insufficient variation for correlation.')
    else:
        print('Too few overlapping days for correlation.')
else:
    print('No defaults or no VBT data — correlation skipped.')

## D. Supplementary Analysis

Trade volume, bid-ask spreads, and inventory-price relationships.

In [ ]:
# Cell 12: D1. Trade Volume — stacked bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Trade Volume by Bucket', fontsize=14, fontweight='bold')

if trade_days:
    # Count and face value by bucket per day
    for ax, metric_fn, title, ylabel in [
        (ax1, lambda trades: len(trades), 'Trade Count', 'Number of trades'),
        (ax2, lambda trades: sum(float(t.face_value) for t in trades), 'Face Value Traded', 'Total face value'),
    ]:
        bottoms = np.zeros(len(trade_days))
        for bucket in BUCKETS:
            vals = [metric_fn(trades_by_day_bucket[d].get(bucket, []))
                    for d in trade_days]
            ax.bar(trade_days, vals, bottom=bottoms, color=BUCKET_COLORS[bucket],
                   label=bucket.capitalize(), alpha=0.8)
            bottoms += np.array(vals)

        ax.set_xlabel('Day')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8)
else:
    for ax in (ax1, ax2):
        ax.text(0.5, 0.5, 'No trades recorded', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: D2. Bid-Ask Spread
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bid-Ask Spread Analysis', fontsize=14, fontweight='bold')

# Left: spread lines by bucket
has_spread = False
for bucket in BUCKETS:
    sp_days = [d for d in all_days if spread_ts.get(d, {}).get(bucket) is not None]
    sp_vals = [spread_ts[d][bucket] for d in sp_days]
    if sp_days:
        ax1.plot(sp_days, sp_vals, color=BUCKET_COLORS[bucket], linewidth=1.5,
                 label=f'{bucket.capitalize()}', marker='.', markersize=4)
        has_spread = True

if has_spread:
    ax1.set_xlabel('Day')
    ax1.set_ylabel('Spread (ask - bid)')
    ax1.set_title('Spread Over Time')
    ax1.legend(fontsize=8)
else:
    ax1.text(0.5, 0.5, 'No spread data', transform=ax1.transAxes,
             ha='center', va='center', fontsize=12, color='gray')

# Right: bid-ask band for short bucket with trade executions
focus_bucket = 'short'
bid_days = [d for d in all_days if dealer_bid_ts.get(d, {}).get(focus_bucket) is not None]
ask_days = [d for d in all_days if dealer_ask_ts.get(d, {}).get(focus_bucket) is not None]
common_ba_days = sorted(set(bid_days) & set(ask_days))

if common_ba_days:
    bids = [dealer_bid_ts[d][focus_bucket] for d in common_ba_days]
    asks = [dealer_ask_ts[d][focus_bucket] for d in common_ba_days]

    ax2.fill_between(common_ba_days, bids, asks, alpha=0.2, color=BUCKET_COLORS[focus_bucket],
                     label='Bid-Ask band')
    ax2.plot(common_ba_days, bids, color=BUCKET_COLORS[focus_bucket], linewidth=1, alpha=0.6)
    ax2.plot(common_ba_days, asks, color=BUCKET_COLORS[focus_bucket], linewidth=1, alpha=0.6)

    # Overlay individual trade executions
    for d in trade_days:
        for t in trades_by_day_bucket[d].get(focus_bucket, []):
            marker = '^' if t.side == 'BUY' else 'v'
            color = 'green' if t.side == 'BUY' else 'red'
            ax2.scatter(t.day, float(t.unit_price), marker=marker, c=color,
                        s=30, zorder=5, alpha=0.7)

    # Add legend entries for trade markers
    ax2.scatter([], [], marker='^', c='green', s=30, label='Buy execution')
    ax2.scatter([], [], marker='v', c='red', s=30, label='Sell execution')

    ax2.set_xlabel('Day')
    ax2.set_ylabel('Price')
    ax2.set_title(f'{focus_bucket.capitalize()} Bucket: Bid-Ask Band + Trades')
    ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, 'No bid-ask data', transform=ax2.transAxes,
             ha='center', va='center', fontsize=12, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: D3. Inventory vs Price
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dealer Inventory vs Midline Price', fontsize=14, fontweight='bold')

for idx, bucket in enumerate(BUCKETS):
    ax = axes[idx]
    color = BUCKET_COLORS[bucket]

    inv_days = [d for d in all_days
                if inventory_ts.get(d, {}).get(bucket) is not None
                and dealer_mid_ts.get(d, {}).get(bucket) is not None]
    inventories = [inventory_ts[d][bucket] for d in inv_days]
    midlines = [dealer_mid_ts[d][bucket] for d in inv_days]

    if inv_days and len(set(inventories)) > 1:
        # Color by day
        norm = Normalize(vmin=min(inv_days), vmax=max(inv_days))
        sc = ax.scatter(inventories, midlines, c=inv_days, cmap='viridis',
                        norm=norm, s=30, alpha=0.7, edgecolors='white', linewidth=0.5)
        plt.colorbar(sc, ax=ax, label='Day')

        # Linear trend
        slope, intercept, _, _, _ = stats.linregress(inventories, midlines)
        x_fit = np.linspace(min(inventories), max(inventories), 50)
        ax.plot(x_fit, slope * x_fit + intercept, 'k--', linewidth=1, alpha=0.6)

        direction = 'higher inv = higher price' if slope > 0 else 'higher inv = lower price'
        ax.text(0.05, 0.95, f'slope={slope:.4f}\n{direction}',
                transform=ax.transAxes, va='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    elif inv_days:
        ax.scatter(inventories, midlines, c=color, s=30, alpha=0.7)
        ax.text(0.05, 0.95, 'Single inventory level\n(no trend)', transform=ax.transAxes,
                va='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='gray')

    ax.set_title(f'{bucket.capitalize()} bucket', fontweight='bold')
    ax.set_xlabel('Dealer Inventory')
    if idx == 0:
        ax.set_ylabel('Dealer Midline')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: Summary Statistics
print('=' * 60)
print('SUMMARY STATISTICS')
print('=' * 60)

summary = metrics.summary()

print(f'\n--- Dealer Profitability ---')
print(f"  Total PnL:          {summary['dealer_total_pnl']:.4f}")
print(f"  Total Return:       {summary['dealer_total_return']:.4f}")
print(f"  Profitable:         {summary['dealer_profitable']}")
print(f"  Spread Income:      {summary['spread_income_total']:.4f}")
print(f"  PnL by Bucket:")
for b, pnl in summary.get('dealer_pnl_by_bucket', {}).items():
    print(f"    {b}: {pnl:.4f}")

print(f'\n--- Trade Counts ---')
print(f"  Total:              {summary['total_trades']}")
print(f"  Buys:               {summary['total_buy_trades']}")
print(f"  Sells:              {summary['total_sell_trades']}")
print(f"  Interior:           {summary['interior_trades']}")
print(f"  Passthrough:        {summary['passthrough_trades']}")

print(f'\n--- Trader Returns ---')
print(f"  Mean Return:        {summary['mean_trader_return']:.4f}")
print(f"  Fraction Profitable:{summary['fraction_profitable_traders']:.4f}")
print(f"  Liquidity Sales:    {summary['liquidity_driven_sales']}")
print(f"  Rescue Events:      {summary['rescue_events']}")

print(f'\n--- System ---')
print(f"  Initial Debt:       {summary['initial_total_debt']:.2f}")
print(f"  Initial Money:      {summary['initial_total_money']:.2f}")
print(f"  Debt/Money Ratio:   {summary['debt_to_money_ratio']:.4f}")
print(f"  Agent Defaults:     {n_defaults}")

print(f'\n--- Final Prices ---')
print(f"  Dealer Mid (final): {summary.get('dealer_mid_final', {})}")
print(f"  VBT Mid (final):    {summary.get('vbt_mid_final', {})}")

print(f'\n--- Repayment Safety ---')
print(f"  Unsafe Buys:        {summary['unsafe_buy_count']}")
print(f"  Fraction Unsafe:    {summary['fraction_unsafe_buys']:.4f}")
mmad = summary.get('mean_margin_at_default')
print(f"  Mean Margin@Default:{f' {mmad:.4f}' if mmad is not None else ' N/A'}")

print('\n' + '=' * 60)